# Fine-Tune Qwen2.5-3B for Arabic Teaching (SageMaker)

**Model:** Qwen2.5-3B-Instruct with LoRA (using Unsloth)

**Training data:** ~115 conversations (teaching, grading, exercise generation)

**Method:** LoRA (rank=32, alpha=32)

**Max sequence length:** 1536 tokens

**Output:** 4-bit quantized model (~2GB)

---

## Setup for SageMaker

1. **Instance Type:** ml.g4dn.xlarge or ml.g5.xlarge (T4/A10G GPU)
2. **HF Token:** Set as environment variable `HF_TOKEN` or in cell below
3. **Training data:** Upload `combined_training_data.jsonl` to EFS at `/home/sagemaker-user/user-default-efs/arabic-teaching/data/`
4. **Run all cells** (~20-30 minutes)
5. **Model output:** Saved to EFS at `/home/sagemaker-user/user-default-efs/arabic-teaching/models/qwen-3b-arabic-teaching/`

## Cell 1: Install Dependencies

In [ ]:
%%capture
# Simple installation - use SageMaker's default PyTorch
!pip install --upgrade torchvision
!pip install unsloth trl transformers datasets peft accelerate bitsandbytes
!pip uninstall -y pyarrow
!pip install pyarrow --no-cache-dir

## ⚠️ RESTART KERNEL - Then run all cells below

## Cell 2: HuggingFace Auth

In [ ]:
import os

from huggingface_hub import login

# Clear local cache to avoid "No space left on device"
!rm -rf ~/.cache/*
!rm -rf /home/sagemaker-user/.cache/*

# Setup EFS paths for all HuggingFace caching
EFS_ROOT = "/home/sagemaker-user/user-default-efs"
os.environ["HF_HOME"] = os.path.join(EFS_ROOT, "hf_cache")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(EFS_ROOT, "hf_cache")
os.environ["HF_DATASETS_CACHE"] = os.path.join(EFS_ROOT, "hf_cache")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

print("📁 Cache setup:")
print(f"   HF_HOME: {os.environ['HF_HOME']}")
print(f"   TRANSFORMERS_CACHE: {os.environ['TRANSFORMERS_CACHE']}")
print(f"   HF_DATASETS_CACHE: {os.environ['HF_DATASETS_CACHE']}")

# Check disk space
print("\n💾 Disk space:")
!df -h /home/sagemaker-user | grep -E "Filesystem|sagemaker"

# Authenticate with HuggingFace
# Option 1: Set HF_TOKEN as environment variable before starting notebook
# Option 2: Uncomment and paste your token here (not recommended for shared notebooks)
# hf_token = "hf_..."

try:
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        raise ValueError("HF_TOKEN not found in environment")
    login(token=hf_token)
    print("\n✅ HuggingFace auth OK")
except Exception as e:
    print(f"\n❌ {e}")
    print("Please set HF_TOKEN environment variable or uncomment and paste token in cell above")

## Cell 3: Imports & Check GPU

In [ ]:
import gc
import json
import os

import torch

# IMPORTANT: Disable Unsloth statistics BEFORE importing Unsloth
os.environ["UNSLOTH_DISABLE_LOG_STATS"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
if torch.cuda.is_available():
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected! Training will be very slow.")

## Cell 4: Configuration

In [ ]:
# EFS paths (adjust if your EFS mount is different)
EFS_ROOT = "/home/sagemaker-user/user-default-efs"
DATA_PATH = f"{EFS_ROOT}/combined_training_data.jsonl"
OUTPUT_DIR = f"{EFS_ROOT}/models/qwen-3b-arabic-teaching"

# Use the non-Unsloth version (Unsloth will patch it automatically)
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 1536
LOAD_IN_4BIT = True

# LoRA config
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Training config
BATCH_SIZE = 2
GRAD_ACCUM = 4
EPOCHS = 10
LR = 2e-4
WARMUP = 0.03

print("=" * 60)
print("🚀 Fine-Tuning Qwen2.5-3B for Arabic Teaching")
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Data: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"Batch: {BATCH_SIZE}, Accum: {GRAD_ACCUM}, Epochs: {EPOCHS}")
print(f"Max Seq Length: {MAX_SEQ_LENGTH}")
print("=" * 60)

# Verify data file exists
if not os.path.exists(DATA_PATH):
    print(f"\n❌ Training data not found at: {DATA_PATH}")
    print("\nPlease upload combined_training_data.jsonl to EFS at:")
    print(f"   {os.path.dirname(DATA_PATH)}/")
else:
    print("\n✅ Training data found")

# Verify EFS cache is setup
print(f"\n📁 HuggingFace cache: {os.environ.get('HF_HOME', 'NOT SET')}")
if os.path.exists(os.environ.get("HF_HOME", "")):
    print("✅ EFS cache directory exists")
else:
    print("❌ EFS cache directory NOT found - run Cell 2 first!")

## Cell 5: Load Training Data

In [ ]:
conversations = []
with open(DATA_PATH, encoding="utf-8") as f:
    for line in f:
        if line.strip():
            conversations.append(json.loads(line))

print(f"✅ Loaded {len(conversations)} conversations")
print(f"   Steps/epoch: {len(conversations) // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"   Total steps: {(len(conversations) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS}")

## Cell 6: Load Model with Unsloth

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
    token=hf_token,
)
print(f"✅ Model loaded: {model.num_parameters() / 1e9:.2f}B params")

## Cell 7: Add LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA: {trainable:,} trainable ({100 * trainable / total:.2f}%)")

## Cell 8: Format Dataset

In [ ]:
formatted_data = [
    {
        "text": tokenizer.apply_chat_template(
            c["messages"], tokenize=False, add_generation_prompt=False
        )
    }
    for c in conversations
]

dataset = Dataset.from_list(formatted_data)
print(f"✅ Dataset: {len(dataset)} examples")

## Cell 9: Setup Trainer

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.cuda.empty_cache()
gc.collect()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_ratio=WARMUP,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
    average_tokens_across_devices=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    args=training_args,
    packing=False,
)

print("✅ Trainer ready")

## Cell 10: Train (~20-30 min on g4dn.xlarge)

In [ ]:
print("🚀 Training...\n")
stats = trainer.train()

print("\n" + "=" * 60)
print("✅ TRAINING COMPLETE!")
print("=" * 60)
print(f"Loss: {stats.training_loss:.4f}")
print(f"Time: {stats.metrics['train_runtime'] / 60:.1f} min")

## Cell 11: Save Model to EFS

In [ ]:
# Cell 11: Save Model to EFS
import os
import tarfile

# Save to EFS (persists across notebook sessions)
lora_output_dir = f"{OUTPUT_DIR}/lora_adapters"
model.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)

print(f"✅ Saved to EFS: {lora_output_dir}")
print("\n📁 Model persisted on EFS at:")
print(f"   {lora_output_dir}")

print("\n📦 Creating downloadable archive...")

# Create tarball in /tmp (accessible for download)
archive_path = "/tmp/qwen-3b-arabic-teaching-lora.tar.gz"
with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(lora_output_dir, arcname="lora_adapters")

archive_size = os.path.getsize(archive_path) / (1024 * 1024)
print(f"✅ Archive created: {archive_path} ({archive_size:.1f} MB)")

print("\n📥 To download:")
print("   1. File browser → /tmp/qwen-3b-arabic-teaching-lora.tar.gz")
print("   2. Right-click → Download")
print("   3. Extract on your machine")

print("\nModel files in archive:")
for file in os.listdir(lora_output_dir):
    file_path = os.path.join(lora_output_dir, file)
    size_mb = os.path.getsize(file_path) / (1024 * 1024)
    print(f"   - {file} ({size_mb:.1f} MB)")

## Cell 12: Test Generation

In [ ]:
# Prepare for inference
FastLanguageModel.for_inference(model)

# Test with a teaching mode prompt
test_messages = [
    {
        "role": "system",
        "content": "You are a supportive Arabic language teacher. Use an encouraging, warm tone.",
    },
    {
        "role": "user",
        "content": """Mode: teaching_vocab

Batch 1 of 3

Words:
- كِتَابٌ (kitaabun) - book
- مَدْرَسَةٌ (madrasatun) - school

Introduce these words with Arabic, transliteration, and English. Remind them flashcards are available. Offer options:
1. Take quiz on this batch
2. Go to next batch
3. See all words

Format with numbered options and mention they can request something else. Be encouraging and clear.""",
    },
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

print("🧪 Testing model generation...\n")
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.7, top_p=0.9, do_sample=True)

full_output = tokenizer.batch_decode(outputs)[0]

# Extract assistant response
if "<|im_start|>assistant" in full_output:
    response = full_output.split("<|im_start|>assistant")[-1]
    response = response.replace("<|im_end|>", "").strip()
elif "assistant" in full_output:
    response = full_output.split("assistant")[-1].strip()
else:
    response = full_output

print("=" * 60)
print("📝 Test Generation:")
print("=" * 60)
print(response)
print("=" * 60)

# Test grading mode
print("\n\n🧪 Testing grading mode...\n")

grading_test = [
    {"role": "system", "content": "You are a precise Arabic grading assistant. Return only JSON."},
    {
        "role": "user",
        "content": """Mode: grading_vocab

Grade this answer:
Word: مَدْرَسَةٌ
Student answer: school
Correct answer: school

Return JSON: {"correct": true/false}""",
    },
]

grading_prompt = tokenizer.apply_chat_template(
    grading_test, tokenize=False, add_generation_prompt=True
)

grading_inputs = tokenizer([grading_prompt], return_tensors="pt").to("cuda")
grading_outputs = model.generate(
    **grading_inputs, max_new_tokens=50, temperature=0.3, do_sample=False
)

grading_full = tokenizer.batch_decode(grading_outputs)[0]
if "<|im_start|>assistant" in grading_full:
    grading_response = grading_full.split("<|im_start|>assistant")[-1]
    grading_response = grading_response.replace("<|im_end|>", "").strip()
else:
    grading_response = (
        grading_full.split("assistant")[-1].strip() if "assistant" in grading_full else grading_full
    )

print("=" * 60)
print("📝 Grading Test:")
print("=" * 60)
print(grading_response)
print("=" * 60)

## Summary

✅ Fine-tuning complete!

**Model saved to EFS:**
- Location: `/home/sagemaker-user/user-default-efs/arabic-teaching/models/qwen-3b-arabic-teaching/lora_adapters/`
- Persists across notebook sessions
- Ready to use with agents

**Model files:**
- `adapter_config.json`
- `adapter_model.safetensors`
- `tokenizer.json`
- `tokenizer_config.json`
- `special_tokens_map.json`

**Next steps:**
1. Model is already on EFS - no download needed
2. Update agent code to load from EFS path
3. Test with multi-agent system